In [1]:
!pip install -U langchain==0.3.27 langchain-core==0.3.74 langchain-community==0.3.27 langchain-text-splitters==0.3.9 langchain-huggingface langchain-groq langchain-astradb pypdf python-dotenv sentence-transformers


  Using cached langchain_huggingface-1.2.2-py3-none-any.whl.metadata (4.0 kB)
  Using cached langchain_groq-1.1.3-py3-none-any.whl.metadata (2.9 kB)
INFO: pip is looking at multiple versions of langchain-huggingface to determine which version is compatible with other requirements. This could take a while.
  Using cached langchain_huggingface-1.2.1-py3-none-any.whl.metadata (3.3 kB)
  Using cached langchain_huggingface-1.2.0-py3-none-any.whl.metadata (2.8 kB)
  Using cached huggingface_hub-0.36.2-py3-none-any.whl.metadata (15 kB)
  Using cached langchain_huggingface-1.1.0-py3-none-any.whl.metadata (2.8 kB)
  Using cached langchain_huggingface-1.0.1-py3-none-any.whl.metadata (2.1 kB)
  Using cached langchain_huggingface-1.0.0-py3-none-any.whl.metadata (2.1 kB)
INFO: pip is looking at multiple versions of langchain-groq to determine which version is compatible with other requirements. This could take a while.
  Using cached langchain_groq-1.1.2-py3-none-any.whl.metadata (2.4 kB)
  Using c

In [2]:
import os

from dotenv import load_dotenv

from langchain_community.document_loaders import PyPDFLoader

from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_huggingface import HuggingFaceEmbeddings

from langchain_astradb import AstraDBVectorStore

from langchain_groq import ChatGroq

from langchain_core.prompts import ChatPromptTemplate

from langchain.chains.combine_documents import create_stuff_documents_chain

from langchain.chains.retrieval import create_retrieval_chain

In [ ]:
load_dotenv()

GROQ_API_KEY = os.getenv("your groq key")

ASTRA_DB_API_ENDPOINT = os.getenv("astra db api")

ASTRA_DB_APPLICATION_TOKEN = os.getenv("token")

In [6]:
loader = PyPDFLoader("New_Resume_Vivek.pdf")

documents = loader.load()

print(len(documents))

2


In [7]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

docs = text_splitter.split_documents(documents)

print(len(docs))

5


In [8]:
embedding = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

c:\Users\Asus\OneDrive\Desktop\GEN AI\Langchain\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 696.68it/s]


In [9]:
vectorstore = AstraDBVectorStore(
    collection_name="pdf_query",

    embedding=embedding,

    api_endpoint=ASTRA_DB_API_ENDPOINT,

    token=ASTRA_DB_APPLICATION_TOKEN,
)

In [10]:
vectorstore.add_documents(docs)

print("Documents Added!")

Documents Added!


In [11]:
retriever = vectorstore.as_retriever(
    search_kwargs={"k":3}
)

In [12]:
llm = ChatGroq(
    groq_api_key=GROQ_API_KEY,
    model_name="llama-3.3-70b-versatile"
)

In [13]:
prompt = ChatPromptTemplate.from_template(
"""
Answer the question based only on the provided context.

<context>
{context}
</context>

Question:
{input}
"""
)

In [14]:
document_chain = create_stuff_documents_chain(
    llm,
    prompt
)

In [15]:
retrieval_chain = create_retrieval_chain(
    retriever,
    document_chain
)

In [16]:
response = retrieval_chain.invoke(
    {
        "input":"What is this PDF about?"
    }
)

print(response["answer"])

This appears to be a resume or a portfolio of Pobbahti Vivek, showcasing their education, skills, experience, and projects, particularly in the fields of computer science, data science, and cybersecurity.


In [17]:
while True:

    question = input("Ask : ")

    if question.lower() == "exit":
        break

    response = retrieval_chain.invoke(
        {
            "input":question
        }
    )

    print("\nAnswer:\n")

    print(response["answer"])

    print("-"*80)


Answer:

This resume appears to be about a computer science engineering student or recent graduate, highlighting their education, skills, projects, and experiences in the field of computer science and cybersecurity.
--------------------------------------------------------------------------------

Answer:

You haven't asked a question yet. Please go ahead and ask your question based on the provided context, and I'll do my best to answer it.
--------------------------------------------------------------------------------

Answer:

It seems like you forgot to include the question. Please go ahead and ask your question based on the provided context, and I'll do my best to answer it.
--------------------------------------------------------------------------------

Answer:

There is no information or question related to "Escape" in the provided context. The context seems to be a resume or a portfolio of a computer science student, highlighting their projects, skills, and education.
--------